In [ ]:
!pip install google-generativeai langchain langchain-core

In [2]:
import google.generativeai as genai
from getpass import getpass
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnableLambda


In [3]:
api_key = getpass("🔑 Enter your Gemini API key: ")
genai.configure(api_key=api_key)

# Create a custom LLM class that wraps Gemini for LangChain compatibility
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Mapping, Any

class GeminiLLM(LLM):
    model_name: str = "gemini-2.5-flash"
    streaming: bool = False
    
    @property
    def _llm_type(self) -> str:
        return "gemini"
    
    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        model = genai.GenerativeModel(self.model_name)
        response = model.generate_content(prompt)
        
        # Simulate streaming if enabled
        if self.streaming:
            for chunk in response.text.split():
                print(chunk, end=" ", flush=True)
            print()
            
        return response.text
    
    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {"model_name": self.model_name}

model = GeminiLLM()
model_stream = GeminiLLM(streaming=True)

## Prompt + Parser

In [4]:
prompt = ChatPromptTemplate.from_template("Write a short motivational quote about {topic}.")
parser = StrOutputParser()

## Create a RunnableSequence

In [5]:
chain = (
    prompt
    | (lambda x: x.to_string())  # Regular function works fine with pipe operator
    | model
    | parser
)

### The output will stream live

In [6]:
result = chain.invoke({"topic": "perseverance"})
print("\nFinal Parsed Result:", result)


Final Parsed Result: **Perseverance: The quiet strength that turns the impossible into the inevitable.**


## Batching Inputs

In [7]:
inputs = [
    {"topic": "courage"},
    {"topic": "teamwork"},
    {"topic": "innovation"}
]

batch_results = chain.batch(inputs)
print(batch_results)

["Courage isn't the absence of fear, but the willingness to move forward despite it. It's the first step to unlocking your true potential.", '"Teamwork: Where individual strengths merge to achieve the extraordinary."', '"Innovation: The courage to see beyond what is and the will to create what can be."']


## RunnableParallel (Run multiple chains at once)

In [10]:
# Create the base chain first using pipe (this works)
base_chain = prompt | (lambda x: x.to_string()) | model | parser

# --- Wrap each branch to pick its input ---
branch1 = RunnableLambda(lambda x: base_chain.invoke(x["branch1"]))
branch2 = RunnableLambda(lambda x: base_chain.invoke(x["branch2"]))

# --- RunnableParallel: dict of branches ---
parallel_chain = RunnableParallel({"branch1": branch1, "branch2": branch2})

# --- Input dict must contain keys for each branch ---
inputs = {
    "branch1": {"topic": "focus"},
    "branch2": {"topic": "happiness"}
}

# --- Invoke parallel chain ---
result_parallel = parallel_chain.invoke(inputs)
print(result_parallel)

{'branch1': '"Your focus defines your reality. Choose it wisely."', 'branch2': '"Happiness blooms when you choose to water your life with gratitude and joy."'}
